# Multimodal PDF search*<br><br>

Handling PDF files, which often contain a mix of text, images, and layout information, presents a challenge for traditional embedding methods. This usually requires a multimodal generative model to pre-process the documents into a format that is suitable for the embedding model. This intermediate text representations can lose critical information; for example, the structure and precise content of tables or complex layouts might not be accurately rendered
<br>

In [1]:
import os

from getpass import getpass
import nest_asyncio

from dotenv import load_dotenv

nest_asyncio.apply()

load_dotenv()
CO_API_KEY = os.environ['CO_API_KEY'] or getpass("Enter your Cohere API key: ")

In [2]:
import cohere
import numpy as np
co = cohere.Client(
    api_key=CO_API_KEY
)  # Get your free API key: https://dashboard.cohere.com/api-keys

In [3]:
from pdf2image import convert_from_path
from io import BytesIO
import base64
import chromadb
import cohere


Embed v5 solves this problem as it is designed to natively understand mixed-modality inputs. Embed v5 can directly process the PDF content, including text and images, in a single step. It generates a unified embedding that captures the semantic meaning derived from both the textual and visual elements.>

In [4]:
pdf_path = "files/FullManual_02_ENG.pdf"  # https://github.com/cohere-ai/cohere-developer-experience/raw/main/notebooks/guides/embed-v4-pdf-search/data/Samsung_Home_Theatre_HW-N950_ZA_FullManual_02_ENG_180809_2.pdf
pages = convert_from_path(pdf_path, dpi=110, thread_count=4)
total_pages = len(pages)

input_array = []
for page in pages:
    buffer = BytesIO()
    page.save(buffer, format="PNG")
    
    base64_str = base64.b64encode(buffer.getvalue()).decode("utf-8")    
    base64_image = f"data:image/png;base64,{base64_str}"

    input_array.append(base64_image)

Next, generate the embeddings for these pages and store them in a vector database (in this example, we use Chroma).

In [21]:
import time

# 2. Resilient Batching Loop (Safe pacing for Trial Keys)
# Generate the document embeddings

BATCH_SIZE = 3  # Small batch to control immediate token volume per request
embeddings = []
# Pacing delay in seconds to ensure you stay under the 100,000 tokens/minute ceiling
TRIAL_COOLDOWN = 12  

print(f"🚀 Safe embedding processing running in batches of {BATCH_SIZE}...")
for i in range(0, total_pages, BATCH_SIZE):
    batch = input_array[i:i + BATCH_SIZE]
    print(f" Processing batch: pages {i+1} to {min(i + BATCH_SIZE, total_pages)}...")
    
    try:
        for j in range(0, len(batch)):
            res = co.embed(
                model="embed-v5.0-pro",
                images=[input_array[i]],
                input_type="search_document",
                embedding_types=["float"]
                )
            batch_vectors = res.embeddings.float[0]
            embeddings.append(batch_vectors)
        
    except cohere.core.api_error.ApiError as e:
        if e.status_code == 429:
            print("⚠️ Rate limit triggered. Waiting 30 seconds to refresh the token pool...")
            time.sleep(30)
            # Retry processing the exact same batch again
            i -= BATCH_SIZE 
            continue
        else:
            print(f"❌ API Error occurred: {e}")
            break
            
    # Add a steady delay between successful iterations to satisfy the minute windows
    if i + BATCH_SIZE < total_pages:
        print(f"😴 Sleeping for {TRIAL_COOLDOWN} seconds to avoid hitting Trial rate limits...")
        time.sleep(TRIAL_COOLDOWN)

print("\n🎉 Process Complete!")
print(f"Total vectors successfully extracted: {len(embeddings)}")



# Store the embeddings in a vector database
ids = []
for i in range(0, len(input_array)):
    ids.append(str(i))


🚀 Safe embedding processing running in batches of 3...
 Processing batch: pages 1 to 3...
😴 Sleeping for 12 seconds to avoid hitting Trial rate limits...
 Processing batch: pages 4 to 6...
😴 Sleeping for 12 seconds to avoid hitting Trial rate limits...
 Processing batch: pages 7 to 9...
😴 Sleeping for 12 seconds to avoid hitting Trial rate limits...
 Processing batch: pages 10 to 12...
😴 Sleeping for 12 seconds to avoid hitting Trial rate limits...
 Processing batch: pages 13 to 15...
😴 Sleeping for 12 seconds to avoid hitting Trial rate limits...
 Processing batch: pages 16 to 18...
😴 Sleeping for 12 seconds to avoid hitting Trial rate limits...
 Processing batch: pages 19 to 21...
😴 Sleeping for 12 seconds to avoid hitting Trial rate limits...
 Processing batch: pages 22 to 24...
😴 Sleeping for 12 seconds to avoid hitting Trial rate limits...
 Processing batch: pages 25 to 27...
😴 Sleeping for 12 seconds to avoid hitting Trial rate limits...
 Processing batch: pages 28 to 30...
😴 Sle

In [23]:

chroma_client = chromadb.Client()
collection = chroma_client.create_collection("pdf_pages_semenov")
collection.add(    
    embeddings=embeddings,
    ids=ids,
)    

Finally, provide a query and run a search over the documents. This will return a list of sorted IDs representing the most similar pages to the query.

In [41]:
query = "Do the speakers come an BT PAIRING Mode?"
# Generate the query embedding
query_embeddings = co.embed(
    model="embed-v5.0-pro",
    input_type="search_query",
    embedding_types=["float"],
    texts=[query],
).embeddings.float[0]
# Search the vector database
results = collection.query(
    query_embeddings=[query_embeddings],
    n_results=5,  # Define the top_k value
)

# Print the id of the top-ranked page
r = results['ids'][0][0]
print(f"The top ranked page is {r} in the document")

The top ranked page is 24 in the document
